In [1]:
"""
Backpropagation explained step-by-step with PyTorch
---------------------------------------------------
This program trains a simple feedforward neural network on the MNIST dataset.
It illustrates:
- Forward propagation
- Loss computation
- Backward propagation (autograd)
- Weight updates
- Manual gradient verification (optional)
"""

'\nBackpropagation explained step-by-step with PyTorch\n---------------------------------------------------\nThis program trains a simple feedforward neural network on the MNIST dataset.\nIt illustrates:\n- Forward propagation\n- Loss computation\n- Backward propagation (autograd)\n- Weight updates\n- Manual gradient verification (optional)\n'

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

In [3]:
# ---------------------------
# 1. Load and preprocess data (open source: MNIST)
# ---------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # mean and std of MNIST
])

# Load full training set
trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                       download=True, transform=transform)
# Use a subset for faster demonstration (first 2000 samples)
trainset = Subset(trainset, range(2000))
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

# Test set (full for evaluation)
testset = torchvision.datasets.MNIST(root='./data', train=False,
                                      download=True, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 46.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.11MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.36MB/s]


In [4]:
# ---------------------------
# 2. Define the neural network architecture
# ---------------------------
class FeedforwardNN(nn.Module):
    """A simple feedforward network with two hidden layers."""
    def __init__(self):
        super(FeedforwardNN, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)   # input layer -> hidden 1
        self.fc2 = nn.Linear(128, 64)       # hidden 1 -> hidden 2
        self.fc3 = nn.Linear(64, 10)        # hidden 2 -> output (10 classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Flatten the image (batch_size, 1, 28, 28) -> (batch_size, 784)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)                     # no softmax: CrossEntropyLoss includes it
        return x

# Instantiate the network, loss function, and optimizer
net = FeedforwardNN()
criterion = nn.CrossEntropyLoss()            # combines LogSoftmax and NLLLoss
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9)

In [5]:
# ---------------------------
# 3. Training loop with backpropagation explained
# ---------------------------
print("Starting training...\n")
for epoch in range(3):                        # loop over the dataset 3 times
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        # ---------- Forward pass ----------
        # Compute predicted outputs by passing inputs to the network
        outputs = net(inputs)                  # (batch_size, 10)
        loss = criterion(outputs, labels)      # scalar tensor

        # ---------- Backward pass (the core of backpropagation) ----------
        # Zero the parameter gradients (otherwise they accumulate)
        optimizer.zero_grad()

        # BACKPROPAGATION STARTS HERE
        # This single line computes the gradient of the loss w.r.t. all parameters
        # using the chain rule (automatic differentiation). Internally, PyTorch:
        #   - traverses the computational graph backwards from loss
        #   - applies the chain rule at each node
        #   - accumulates gradients in each parameter's .grad attribute
        loss.backward()

        # (Optional) Inspect gradients of the first layer's weights
        if i == 0 and epoch == 0:
            print("\nAfter first backward pass:")
            print(f"  Gradient of fc1.weight : mean={net.fc1.weight.grad.mean():.6f}, "
                  f"std={net.fc1.weight.grad.std():.6f}")
            print(f"  Gradient of fc1.bias   : mean={net.fc1.bias.grad.mean():.6f}, "
                  f"std={net.fc1.bias.grad.std():.6f}")

        # ---------- Weight update ----------
        # Update parameters using the computed gradients (SGD step)
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        if i % 100 == 99:                       # print every 100 mini-batches
            print(f"Epoch {epoch+1}, batch {i+1:3d}  loss: {running_loss/100:.4f}")
            running_loss = 0.0

print("\nFinished training.\n")

Starting training...


After first backward pass:
  Gradient of fc1.weight : mean=0.000019, std=0.002583
  Gradient of fc1.bias   : mean=0.000257, std=0.002689

Finished training.



In [6]:
# ---------------------------
# 4. Evaluate the network on test data
# ---------------------------
correct = 0
total = 0
with torch.no_grad():                           # disable gradient tracking
    for inputs, labels in testloader:
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)    # get the class with highest score
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test accuracy: {100 * correct / total:.2f}%")

# ---------------------------
# 5. BONUS: Manual backpropagation for one batch (to demystify autograd)
# ---------------------------
print("\n" + "="*60)
print("MANUAL BACKPROPAGATION VERIFICATION (for one batch)")
print("="*60)

# Take a single batch
data_iter = iter(trainloader)
images, labels = next(data_iter)

# ---- Forward pass manually ----
# We'll use the same network but detach to avoid interfering with the trained model
x = images.view(images.size(0), -1).detach().requires_grad_(True)

# Manually compute each layer (keeping intermediate values)
z1 = x @ net.fc1.weight.T + net.fc1.bias        # linear 1
a1 = torch.relu(z1)                              # activation 1
z2 = a1 @ net.fc2.weight.T + net.fc2.bias        # linear 2
a2 = torch.relu(z2)                              # activation 2
z3 = a2 @ net.fc3.weight.T + net.fc3.bias        # linear 3 (logits)

# Loss: cross-entropy (simplified: we'll use PyTorch's function)
loss_manual = criterion(z3, labels)

# ---- Backward pass manually (gradient of loss w.r.t. fc1.weight) ----
# Let's compute dL/dW1 using the chain rule:
# dL/dW1 = (dL/dz3) * (dz3/da2) * (da2/dz2) * (dz2/da1) * (da1/dz1) * (dz1/dW1)

# First, get the gradient of loss w.r.t. z3 (output logits) manually:
# For cross-entropy with softmax, dL/dz3 = softmax(z3) - y_onehot
softmax = torch.exp(z3) / torch.exp(z3).sum(dim=1, keepdim=True)
y_onehot = torch.eye(10)[labels]                 # convert labels to one-hot
dL_dz3 = (softmax - y_onehot) / z3.shape[0]      # divide by batch size (as in criterion)

# Backprop through fc3: dL/dW3 = a2.T @ dL/dz3, dL/db3 = sum(dL/dz3, dim=0)
dL_da2 = dL_dz3 @ net.fc3.weight                  # gradient w.r.t a2

# Through ReLU of second layer: dL/dz2 = dL/da2 * 1{z2>0}
dL_dz2 = dL_da2.clone()
dL_dz2[z2 <= 0] = 0                               # ReLU derivative

# Backprop through fc2: dL/dW2 = a1.T @ dL/dz2
dL_da1 = dL_dz2 @ net.fc2.weight

# Through ReLU of first layer: dL/dz1 = dL/da1 * 1{z1>0}
dL_dz1 = dL_da1.clone()
dL_dz1[z1 <= 0] = 0

# Finally, dL/dW1 = x.T @ dL/dz1
dL_dW1_manual = x.T @ dL_dz1

# Compare with PyTorch's autograd computed gradient for the same batch
# We need to hook into the existing network's gradients. Since we already trained,
# we'll temporarily zero gradients and run a forward/backward on this batch.
net.zero_grad()
outputs = net(images)                             # forward using the trained net
loss = criterion(outputs, labels)
loss.backward()                                   # autograd computes gradients
dL_dW1_auto = net.fc1.weight.grad.clone()

# Compare
print("\nGradient of fc1.weight (first 5x5 corner):")
print("  Autograd  :\n", dL_dW1_auto[:5, :5])
print("  Manual    :\n", dL_dW1_manual[:5, :5])
print("\nMax absolute difference:",
      torch.max(torch.abs(dL_dW1_auto - dL_dW1_manual)).item())

# Show that they are essentially identical (up to floating point)
if torch.allclose(dL_dW1_auto, dL_dW1_manual, rtol=1e-5):
    print("✓ Manual gradients match PyTorch autograd! Backpropagation works.")
else:
    print("✗ Discrepancy – check your manual derivation.")

Test accuracy: 85.18%

MANUAL BACKPROPAGATION VERIFICATION (for one batch)

Gradient of fc1.weight (first 5x5 corner):
  Autograd  :
 tensor([[-0.0020, -0.0020, -0.0020, -0.0020, -0.0020],
        [-0.0005, -0.0005, -0.0005, -0.0005, -0.0005],
        [ 0.0027,  0.0027,  0.0027,  0.0027,  0.0027],
        [ 0.0023,  0.0023,  0.0023,  0.0023,  0.0023],
        [ 0.0010,  0.0010,  0.0010,  0.0010,  0.0010]])
  Manual    :
 tensor([[-0.0020, -0.0005,  0.0027,  0.0023,  0.0010],
        [-0.0020, -0.0005,  0.0027,  0.0023,  0.0010],
        [-0.0020, -0.0005,  0.0027,  0.0023,  0.0010],
        [-0.0020, -0.0005,  0.0027,  0.0023,  0.0010],
        [-0.0020, -0.0005,  0.0027,  0.0023,  0.0010]],
       grad_fn=<SliceBackward0>)


RuntimeError: The size of tensor a (784) must match the size of tensor b (128) at non-singleton dimension 1